In [ ]:
from mistocr.core import ocr, read_pgs
from mistocr.refine import fix_md_hdgs, fmt_hdgs_idx

from re import findall, MULTILINE
from toolslm.md_hier import create_heading_dict
from functools import reduce

## What is `mistocr`

`mistocr` is a Python library that provides simple batch OCR for PDFs using Mistral's state-of-the-art vision model. It's designed for processing large document collections efficiently.

Key advantages over tools like pdfplumber:

- **Performance**: Mistral's OCR delivers state-of-the-art accuracy on complex documents including tables, charts, and multi-column layouts
- **Scale**: Process entire folders of PDFs in a single batch job, with asynchronous processing
- **Cost savings**: Batch OCR mode reduces costs from $1/1000 pages to $0.50/1000 pages
- **Natively multilingual and multimodal**, supporting thousands of scripts and languages
- **Utility functions**: The library includes a `refine` module with utilities that help you process and enrich headings in the extracted markdown as well as describing images/figures of the pdf

## How to use `mistocr`


To process a single `pdf`file, let' say the ["Resnet" paper](https://arxiv.org/abs/1512.03385):

*Note: you can also process a bunch of them at once (we use `batch` processing by default)*

In [ ]:
pdf_fname = '../files/test/resnet.pdf'


Just specify the destination folder (here `md`):

In [ ]:
results = ocr(pdf_fname, 'md')
results

(#1) [Path('md/resnet')]

And you'll get each page of the paper in a specific markdown file as well as an `img` folder containing figures.

In [ ]:
!ls -R md

md:
resnet

md/resnet:
img	   page_10.md  page_12.md  page_3.md  page_5.md  page_7.md  page_9.md
page_1.md  page_11.md  page_2.md   page_4.md  page_6.md  page_8.md

md/resnet/img:
img-0.jpeg  img-2.jpeg	img-4.jpeg  img-6.jpeg
img-1.jpeg  img-3.jpeg	img-5.jpeg


Let's read them all in a single `md` file:

In [ ]:
md = read_pgs('md/resnet')

In [ ]:
print(md[:200])

# Deep Residual Learning for Image Recognition 

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


#### Abstract

Deeper neur


Let's take a quick look at its headings:

In [ ]:
get_hdgs = lambda o: findall(r'^#{1,6} .+$', o, MULTILINE)

In [ ]:
get_hdgs(md)

['# Deep Residual Learning for Image Recognition ',
 '#### Abstract',
 '## 1. Introduction',
 '## 2. Related Work',
 '## 3. Deep Residual Learning',
 '### 3.1. Residual Learning',
 '### 3.2. Identity Mapping by Shortcuts',
 '### 3.3. Network Architectures',
 '### 3.4. Implementation',
 '## 4. Experiments',
 '### 4.1. ImageNet Classification',
 '### 4.2. CIFAR-10 and Analysis',
 '### 4.3. Object Detection on PASCAL and MS COCO',
 '## References',
 '## A. Object Detection Baselines',
 '## PASCAL VOC',
 '## MS COCO',
 '## B. Object Detection Improvements',
 '## MS COCO',
 '## PASCAL VOC',
 '## ImageNet Detection',
 '## C. ImageNet Localization']

### Fixing the headings

Looking at the headings, the OCR did a reasonable job capturing the document structure. However, the "Abstract" being marked as level 4 (####) seems inconsistent with the rest of the hierarchy - it would make more sense as a level 2 (##) heading to match the other major sections like "Introduction" and "Related Work" for instance.

In my experience, when working with PDFs in the wild, I'll typically find documents that are far messier than this LaTeX-generated paper, especially larger files with inconsistent formatting.

To fix this, mistocr provides a refine module with utilities such as `fix_md_hdgs` that will ask an LLM to fix the headings and replace them in the original document:

In [ ]:
fix_md_hdgs('md/resnet', dst='md_fixed/resnet')

In [ ]:
md_fixed = read_pgs('md_fixed/resnet')


In [ ]:
get_hdgs(md_fixed)

['# Deep Residual Learning for Image Recognition  ... page 1',
 '## Abstract ... page 1',
 '## 1. Introduction ... page 1',
 '## 2. Related Work ... page 2',
 '## 3. Deep Residual Learning ... page 3',
 '### 3.1. Residual Learning ... page 3',
 '### 3.2. Identity Mapping by Shortcuts ... page 3',
 '### 3.3. Network Architectures ... page 3',
 '### 3.4. Implementation ... page 4',
 '## 4. Experiments ... page 4',
 '### 4.1. ImageNet Classification ... page 4',
 '### 4.2. CIFAR-10 and Analysis ... page 7',
 '### 4.3. Object Detection on PASCAL and MS COCO ... page 8',
 '## References ... page 9',
 '## A. Object Detection Baselines ... page 10',
 '### PASCAL VOC ... page 10',
 '#### MS COCO ... page 10',
 '## B. Object Detection Improvements ... page 10',
 '#### MS COCO ... page 10',
 '### PASCAL VOC ... page 11',
 '### ImageNet Detection ... page 11',
 '## C. ImageNet Localization ... page 12']

The headings have been significantly improved. Most notably, "Abstract" is now correctly formatted as a level 2 heading, matching other major sections like "Introduction" and "Related Work."

The function also nested subsections appropriately - for example, "PASCAL VOC" now appears under Appendices A and B, though the nesting isn't perfect throughout. Additionally, page numbers have been automatically appended to all headings (e.g., "... page 1"), which can be useful for downstream tasks like RAG applications.

If you'd like to refine the heading structure further, you can pass a custom prompt to fix_md_hdgs. Check the documentation for details on how to customize this behavior.

## Leverage a well-structure markdown

**Reading Documents Like Humans Do**

Traditional RAG systems break documents into chunks, embed them, and retrieve pieces by similarity—essentially handing the LLM a jumbled plate of information and asking it to make sense of the chaos. But that's not how humans read.

When you pick up a research paper, you don't start with random paragraphs. You scan the title, read the abstract, maybe jump to the conclusion, then selectively drill into sections that matter. You navigate the document's *structure* to efficiently extract what you need.

**Can we teach machines to read the same way?**

With well-structured markdown—complete with a proper heading hierarchy—we can. Instead of semantic similarity searches that ignore document organization, we can:

- Start with high-level sections (title, abstract, conclusions)
- Progressively drill down into relevant subsections
- Maintain context about where information sits in the document's logical flow
- Navigate hierarchically rather than randomly

This mirrors human reading strategies and respects the author's intentional structure. The document becomes a navigable map, not a bag of chunks.

**The Evaluation Advantage**

When it's time to evaluate your LLM system and collect feedback from domain experts, this approach shines. Instead of showing them "the model retrieved chunks 47, 203, and 89," you can present a clear trace: "The system read the abstract, then navigated to Section 3.2 on Identity Mapping." This is immediately understandable—both for domain experts reviewing the system and for you when debugging or improving it.

In [ ]:
hdgs = create_heading_dict(md)
hdgs


{'Attention Is All You Need': {'Illia Polosukhin* ${ }^{\\ddagger}$<br>illia.polosukhin@gmail.com': {'Abstract': {}}},
 '1 Introduction': {'2 Background': {}, '3 Model Architecture': {}},
 '3.1 Encoder and Decoder Stacks': {'3.2 Attention': {}},
 '3.2.1 Scaled Dot-Product Attention': {'3.2.2 Multi-Head Attention': {}},
 '3.2.3 Applications of Attention in our Model': {'3.3 Position-wise Feed-Forward Networks': {},
  '3.4 Embeddings and Softmax': {}},
 '3.5 Positional Encoding': {'4 Why Self-Attention': {}},
 '5 Training': {'5.1 Training Data and Batching': {},
  '5.2 Hardware and Schedule': {},
  '5.3 Optimizer': {},
  '5.4 Regularization': {}},
 '6 Results': {'6.1 Machine Translation': {}, '6.2 Model Variations': {}},
 '6.3 English Constituency Parsing': {},
 '7 Conclusion': {'References': {}},
 'Attention Visualizations': {}}

In [ ]:
print(fmt_hdgs_idx(hdgs))

0. Attention Is All You Need
1. 1 Introduction
2. 3.1 Encoder and Decoder Stacks
3. 3.2.1 Scaled Dot-Product Attention
4. 3.2.3 Applications of Attention in our Model
5. 3.5 Positional Encoding
6. 5 Training
7. 6 Results
8. 6.3 English Constituency Parsing
9. 7 Conclusion
10. Attention Visualizations


In [ ]:
def flatten_sections(
    hdgs: dict, # The nested dictionary-like structure of the report also allowing to pull content from 
    path: list = [] # The current path in the nested structure
    ) -> list: # The flat list of (key, full_path) tuples
    "Extract flat list of (key, full_path) tuples from nested hdgs"
    sections = []
    for key, value in hdgs.items():
        current_path = path + [key]
        sections.append((key, current_path))
        if isinstance(value, dict):
            sections.extend(flatten_sections(value, current_path))
    return sections

In [ ]:
flatten_sections(hdgs)

[('Attention Is All You Need', ['Attention Is All You Need']),
 ('Illia Polosukhin* ${ }^{\\ddagger}$<br>illia.polosukhin@gmail.com',
  ['Attention Is All You Need',
   'Illia Polosukhin* ${ }^{\\ddagger}$<br>illia.polosukhin@gmail.com']),
 ('Abstract',
  ['Attention Is All You Need',
   'Illia Polosukhin* ${ }^{\\ddagger}$<br>illia.polosukhin@gmail.com',
   'Abstract']),
 ('1 Introduction', ['1 Introduction']),
 ('2 Background', ['1 Introduction', '2 Background']),
 ('3 Model Architecture', ['1 Introduction', '3 Model Architecture']),
 ('3.1 Encoder and Decoder Stacks', ['3.1 Encoder and Decoder Stacks']),
 ('3.2 Attention', ['3.1 Encoder and Decoder Stacks', '3.2 Attention']),
 ('3.2.1 Scaled Dot-Product Attention',
  ['3.2.1 Scaled Dot-Product Attention']),
 ('3.2.2 Multi-Head Attention',
  ['3.2.1 Scaled Dot-Product Attention', '3.2.2 Multi-Head Attention']),
 ('3.2.3 Applications of Attention in our Model',
  ['3.2.3 Applications of Attention in our Model']),
 ('3.3 Position-wise 

In [ ]:
def format_toc_for_llm(hdgs: dict) -> str:
    """Format ToC as readable text with page numbers"""
    sections = flatten_sections(hdgs)
    lines = [f"- {key}" for key, path in sections]
    return '\n'.join(lines)

In [ ]:
print(format_toc_for_llm(hdgs))

- Attention Is All You Need
- Illia Polosukhin* ${ }^{\ddagger}$<br>illia.polosukhin@gmail.com
- Abstract
- 1 Introduction
- 2 Background
- 3 Model Architecture
- 3.1 Encoder and Decoder Stacks
- 3.2 Attention
- 3.2.1 Scaled Dot-Product Attention
- 3.2.2 Multi-Head Attention
- 3.2.3 Applications of Attention in our Model
- 3.3 Position-wise Feed-Forward Networks
- 3.4 Embeddings and Softmax
- 3.5 Positional Encoding
- 4 Why Self-Attention
- 5 Training
- 5.1 Training Data and Batching
- 5.2 Hardware and Schedule
- 5.3 Optimizer
- 5.4 Regularization
- 6 Results
- 6.1 Machine Translation
- 6.2 Model Variations
- 6.3 English Constituency Parsing
- 7 Conclusion
- References
- Attention Visualizations


In [ ]:
def find_section_path(
    hdgs: dict, # The nested dictionary structure
    target_section: str # The section name to find
) -> list: # The nested key path for the given section name
    "Find the nested key path for a given section name."
    def search_recursive(current_dict, path=[]):
        for key, value in current_dict.items():
            current_path = path + [key]
            if key == target_section:
                return current_path
            if isinstance(value, dict):
                result = search_recursive(value, current_path)
                if result:
                    return result
        return None
    
    return search_recursive(hdgs)

In [ ]:
def get_content_tool(
    hdgs: dict, # The nested dictionary structure
    keys_list: list, # The list of keys to navigate through
    ) -> str: # The content of the section
    "Navigate through nested levels using the exact key strings."
    return reduce(lambda current, key: current[key], keys_list, hdgs).text


In [ ]:
get_content_tool(hdgs, '5 Training')

KeyError: '5'

In [ ]:
# 
def paper(name:str): 
    "Retrieve relevant paper by name"
    return hdgs[name].text